# Stickman Motion Studio — real image-generation smoke test

This is an independent end-to-end test: install ComfyUI, download a pinned Stable Diffusion 1.5 FP16 checkpoint, generate one 512×512 image through the ComfyUI HTTP API, save the PNG and a machine-readable report, then display the image for human inspection.

Model: `Comfy-Org/stable-diffusion-1.5-archive` — `v1-5-pruned-emaonly-fp16.safetensors`, pinned to commit `82ddc01b8d77adc5b4a7c6b5e8f2ddf1a6fe0c08`. Model card identifies CreativeML OpenRAIL-M; review the licence and its use restrictions before production use. The file is 2.13 GB. This is a baseline SD 1.5 test model, not the final Stickman character model.


In [ ]:
import os, sys, subprocess, pathlib, json, time, urllib.request, urllib.error, urllib.parse, hashlib, requests, re, gc
from IPython.display import display, Image as IPyImage
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Kaggle GPU/CUDA is not available.'
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('CUDA devices:', gpu_names)
repo = pathlib.Path('/kaggle/working/ComfyUI')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(repo)], check=True)


## Install ComfyUI without replacing Kaggle PyTorch


In [ ]:
requirements = (repo / 'requirements.txt').read_text(encoding='utf-8').splitlines()
skip = {'torch', 'torchvision', 'torchaudio'}
filtered = []
for line in requirements:
    stripped = line.strip()
    if not stripped or stripped.startswith('#'):
        continue
    match = re.match(r'^([A-Za-z0-9_.-]+)', stripped)
    package_name = match.group(1).lower().replace('_', '-') if match else ''
    if package_name not in skip:
        filtered.append(line)
req_path = pathlib.Path('/kaggle/working/comfyui-image-test-requirements.txt')
req_path.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(req_path)], check=True)
assert torch.cuda.is_available(), 'CUDA was lost during ComfyUI dependency installation.'
print('ComfyUI dependencies installed. PyTorch:', torch.__version__)


## Download and verify the pinned FP16 checkpoint

The download is stored in Kaggle working storage, not committed to GitHub. Its SHA-256 is checked against the model repository's published file metadata.


In [ ]:
model_repo = 'Comfy-Org/stable-diffusion-v1-5-archive'
model_revision = '82ddc01b8d77adc5b4a7c6b5e8f2ddf1a6fe0c08'
model_name = 'v1-5-pruned-emaonly-fp16.safetensors'
model_url = f'https://huggingface.co/{model_repo}/resolve/{model_revision}/{model_name}'
expected_sha256 = 'e9476a13728cd75d8279f6ec8bad753a66a1957ca375a1464dc63b37db6e3916'
checkpoint_dir = repo / 'models' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / model_name
download_started = time.monotonic()
if not checkpoint_path.exists() or checkpoint_path.stat().st_size < 2_000_000_000:
    part_path = checkpoint_path.with_suffix('.safetensors.part')
    with requests.get(model_url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', '0'))
        downloaded = 0
        with part_path.open('wb') as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if downloaded % (256 * 1024 * 1024) < 8 * 1024 * 1024:
                        print(f'Downloaded {downloaded/1e9:.2f} GB' + (f' / {total/1e9:.2f} GB' if total else ''))
    part_path.replace(checkpoint_path)
download_seconds = round(time.monotonic() - download_started, 2)
assert checkpoint_path.stat().st_size > 2_000_000_000, f'Checkpoint file looks incomplete: {checkpoint_path.stat().st_size} bytes'
sha = hashlib.sha256()
with checkpoint_path.open('rb') as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
        sha.update(chunk)
actual_sha256 = sha.hexdigest()
print('Checkpoint bytes:', checkpoint_path.stat().st_size, '| SHA-256:', actual_sha256, '| download seconds:', download_seconds)
assert actual_sha256 == expected_sha256, f'Checkpoint SHA-256 mismatch: {actual_sha256}'


## Start ComfyUI and verify its API


In [ ]:
log_path = pathlib.Path('/kaggle/working/comfyui-image-generation-server.log')
log_file = log_path.open('w')
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'], cwd=repo, stdout=log_file, stderr=subprocess.STDOUT)
base = 'http://127.0.0.1:8188'
stats = None
try:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        if server.poll() is not None:
            break
        try:
            with urllib.request.urlopen(base + '/system_stats', timeout=5) as r:
                stats = json.loads(r.read().decode())
            break
        except Exception:
            time.sleep(3)
    if stats is None:
        log_file.flush()
        print(log_path.read_text(encoding='utf-8', errors='replace')[-15000:])
        raise RuntimeError('ComfyUI did not become ready.')
    with urllib.request.urlopen(base + '/object_info', timeout=60) as r:
        object_info = json.loads(r.read().decode())
    print('ComfyUI ready. Node types:', len(object_info), '| Version:', stats.get('system', {}).get('comfyui_version'))
    assert all(n in object_info for n in ['CheckpointLoaderSimple','CLIPTextEncode','EmptyLatentImage','KSampler','VAEDecode','SaveImage']), 'Required ComfyUI nodes are missing.'


## Generate one image through the ComfyUI API

Fixed seed and prompt make this run reproducible. The prompt intentionally requests a simple flat 2D stickman astronaut illustration suitable for the channel's visual direction.


In [ ]:
prompt_text = 'A friendly simple white stickman astronaut with a perfectly round white head, thick clean dark charcoal outline, tiny expressive black eyes, standing on a small moon surface, one hand waving, Earth visible in the distant sky, flat 2D vector cartoon illustration, clean bold shapes, soft muted cream and blue colours, minimal background, professional animated explainer art, centered full body composition, crisp edges'
negative_text = 'photorealistic, 3d render, realistic human, extra limbs, extra fingers, malformed hands, blurry, noisy, text, watermark, logo, cluttered background, cropped character'
seed = 20261031
width, height = 512, 512
workflow = {
 '1': {'class_type':'CheckpointLoaderSimple','inputs':{'ckpt_name':model_name}},
 '2': {'class_type':'CLIPTextEncode','inputs':{'text':prompt_text,'clip':['1',1]}},
 '3': {'class_type':'CLIPTextEncode','inputs':{'text':negative_text,'clip':['1',1]}},
 '4': {'class_type':'EmptyLatentImage','inputs':{'width':width,'height':height,'batch_size':1}},
 '5': {'class_type':'KSampler','inputs':{'model':['1',0],'positive':['2',0],'negative':['3',0],'latent_image':['4',0],'seed':seed,'steps':20,'cfg':7.0,'sampler_name':'euler','scheduler':'normal','denoise':1.0}},
 '6': {'class_type':'VAEDecode','inputs':{'samples':['5',0],'vae':['1',2]}},
 '7': {'class_type':'SaveImage','inputs':{'images':['6',0],'filename_prefix':'stickman_motion_studio_smoke_test'}}
}
torch.cuda.reset_peak_memory_stats()
generation_started = time.monotonic()
payload = json.dumps({'prompt':workflow,'client_id':'stickman-motion-studio-kaggle-smoke-test'}).encode()
request = urllib.request.Request(base + '/prompt', data=payload, headers={'Content-Type':'application/json'})
with urllib.request.urlopen(request, timeout=60) as r:
    submitted = json.loads(r.read().decode())
prompt_id = submitted['prompt_id']
print('Queued ComfyUI prompt:', prompt_id)
history = None
deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    try:
        with urllib.request.urlopen(base + '/history/' + prompt_id, timeout=20) as r:
            history_response = json.loads(r.read().decode())
        if prompt_id in history_response:
            history = history_response[prompt_id]
            if history.get('outputs'):
                break
            if history.get('status', {}).get('status_str') == 'error':
                raise RuntimeError('ComfyUI prompt execution failed: ' + json.dumps(history.get('status')))
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            raise
    time.sleep(3)
assert history and history.get('outputs'), 'Timed out waiting for generated image.'
generation_seconds = round(time.monotonic() - generation_started, 2)
saved_images = []
for node_output in history['outputs'].values():
    for item in node_output.get('images', []):
        saved_images.append(item)
assert saved_images, 'ComfyUI history has no saved image output.'
image_meta = saved_images[0]
query = urllib.parse.urlencode({'filename':image_meta['filename'],'subfolder':image_meta.get('subfolder',''),'type':image_meta.get('type','output')})
with urllib.request.urlopen(base + '/view?' + query, timeout=60) as r:
    image_bytes = r.read()
output_image = pathlib.Path('/kaggle/working/stickman-motion-studio-generated.png')
output_image.write_bytes(image_bytes)
assert output_image.stat().st_size > 10_000, 'Generated image file is unexpectedly small.'
from PIL import Image
with Image.open(output_image) as im:
    image_dimensions = list(im.size)
    image_format = im.format
assert image_dimensions == [width,height], f'Unexpected image dimensions: {image_dimensions}'
peak_allocated = int(torch.cuda.max_memory_allocated())
peak_reserved = int(torch.cuda.max_memory_reserved())
print('Generation seconds:', generation_seconds, '| image bytes:', output_image.stat().st_size, '| dimensions:', image_dimensions)
print('Peak CUDA allocated/reserved GB:', round(peak_allocated/1e9,2), round(peak_reserved/1e9,2))
display(IPyImage(filename=str(output_image)))


## Save the machine-readable report and server log

The report is only a technical smoke-test pass if every required field is true and the PNG is present. Visual quality and brand consistency still require human review of the displayed image.


In [ ]:
report = {
 'test_name':'comfyui_real_image_generation_smoke_test',
 'comfyui_started':True,
 'api_system_stats_ok':True,
 'api_object_info_ok':True,
 'required_nodes_present':True,
 'image_generation_completed':True,
 'image_saved':output_image.exists(),
 'image_dimensions':image_dimensions,
 'image_format':image_format,
 'image_bytes':output_image.stat().st_size,
 'output_file':output_image.name,
 'prompt_id':prompt_id,
 'seed':seed,
 'steps':20,
 'cfg':7.0,
 'prompt':prompt_text,
 'negative_prompt':negative_text,
 'model_repository':model_repo,
 'model_revision':model_revision,
 'model_filename':model_name,
 'model_source_url':model_url,
 'model_license':'CreativeML OpenRAIL-M (verify terms before production use)',
 'model_bytes':checkpoint_path.stat().st_size,
 'model_sha256':actual_sha256,
 'model_sha256_verified':actual_sha256 == expected_sha256,
 'model_download_seconds':download_seconds,
 'generation_seconds':generation_seconds,
 'torch_version':torch.__version__,
 'cuda_available':torch.cuda.is_available(),
 'gpu_names':gpu_names,
 'cuda_peak_memory_allocated_bytes':peak_allocated,
 'cuda_peak_memory_reserved_bytes':peak_reserved,
 'comfyui_version':stats.get('system',{}).get('comfyui_version'),
 'node_type_count':len(object_info),
 'system_stats':stats,
}
report_path = pathlib.Path('/kaggle/working/image-generation-smoke-test.json')
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
log_file.flush()
log_file.close()
print(json.dumps({k:v for k,v in report.items() if k != 'system_stats'}, indent=2))
print('Saved:', report_path, 'and', log_path)


In [ ]:
required_true = ['comfyui_started','api_system_stats_ok','api_object_info_ok','required_nodes_present','image_generation_completed','image_saved','model_sha256_verified','cuda_available']
failed = [key for key in required_true if report.get(key) is not True]
assert not failed, 'Smoke-test report failed: ' + ', '.join(failed)
assert report['image_dimensions'] == [512,512] and report['image_bytes'] > 10000
print('PASS: ComfyUI generated and saved one real 512x512 image; checkpoint hash and report checks passed.')
